# Python Data Management Workshop

There are numerous ways to set up and work with Python for data science, but Colab is usually the easiest and most convenient.

## What is Google Colab?
Google Colab lets you write and run Python code in your browser — no installation needed. It runs on Google's servers (Linux systems), so your laptop doesn't do the heavy lifting. It is based on Jupyter Notebook, and you can download your code as .ipynb (Jupyter Notebook) files or .py (Python script) files. Colab is ideal for teaching and for smaller projects, but you can also work with .ipynb or .py files locally on your own computer with the proper setup installed.

## Starting Your Own Notebook
When you're ready to start a project from scratch, you have a few options:
- Go to [colab.research.google.com](https://colab.research.google.com) and click **New notebook**
- From Google Drive, click **New → More → Google Colaboratory**
- From any open notebook (including this one), go to **File → New notebook in Drive**

## Working with Files
You'll often need to bring data into your Colab session. There are two main ways:

**Option 1 — Upload a file directly:**
```python
from google.colab import files
files.upload()  
# A dialog will appear to choose your file.
# Alternatively, click the folder icon on the left, then the upload icon.

# With Option 1, you can reference the file just by its filename as a string,
# no path needed, because it is placed in  /content/ and this is the default working directory.
# So for example:

import pandas as pd
df = pd.read_csv('mydata.csv')
```

**Option 2 — Mount your Google Drive:**
```python
from google.colab import drive
drive.mount('/content/drive')
# Your Drive will be accessible at /content/drive/MyDrive/

# You will need to type the full path to your file within your MyDrive directory.
```

## An Important Thing Keep in Mind
Colab sessions are **temporary**. If your session times out or you close the tab, uploaded files, and any files generated during your session that you did not save, will be gone. Mounting your Drive or saving outputs back to Drive is the safest way to keep your work.

## Load any needed modules
Typically it is best to have all the module import statements in the first code block at the top of your notebook file, so they are organized together and we see right away if any packages need to be installed, by running the first cell. But for this workshop I will also import modules as they are introduced.

In [ ]:
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

# If you need a module that is not in Colab, you can install it.
# Google might make changes. They no longer install`datascience` package.
# If you were to need it (not for this workshop):
# !pip install datascience
# from datascience import *

## The Data: Behavioral Risk Factor Surveillance System (BRFSS)

The [**Behavioral Risk Factor Surveillance System (BRFSS)**](https://www.cdc.gov/brfss/) is a CDC-administered telephone survey that has been collecting health-related data on U.S. adults since 1984. It is the largest continuously conducted health survey system in the world, completing over 400,000 interviews per year across all 50 states, the District of Columbia, and several U.S. territories.

The survey captures information on:
- Health risk behaviors (e.g., smoking, alcohol use, physical activity)
- Chronic health conditions (e.g., hypertension, obesity)
- Use of preventive services


### How the BRFSS Survey Is Structured

BRFSS has three layers of content:

**1. The Core Questionnaire** — Asked identically in all states, every year, with no variation. This includes the foundational health topics: chronic conditions, health status, health care access, physical activity, tobacco use, and critically for your project, the **disability module**, which has been part of the core since 2016. All states must ask these questions.

**2. Optional Modules** — Pre-designed question sets that states can elect to add. Examples include modules on falls prevention, pre-diabetes, caregiver, e-cigarette use, etc. A state either opts in or doesn't. This is why coverage varies.

**3. State-Added Questions** — Completely custom questions a state adds on their own. Highly variable and not standardized across states.

We will work with the **Core survey from 2023 and 2024 (two data files)**. Kentucky and Pennsylvania are missing from 2023, and Tennessee is missing from 2024 (not enough data to meet minimum requirements). So combining the two years will enable covering all states.

### A Note on Survey Design

BRFSS uses a complex sampling design, not a simple random sample. Respondents are reached via both landline and cellular telephone, and the sample is stratified geographically. Because of this design, raw counts alone can be misleading: a respondent in a sparsely sampled region represents more people in the population than one from a heavily sampled area.

To account for this, each record in the dataset includes a **survey weight** (`_LLCPWT` for the combined landline + cell dataset). Proper analysis requires specifying the stratification variable (`_STSTR`), the clustering variable (`_PSU`), and the appropriate weight. It is important to keep the survey design in mind when exploring and summarizing the data. However, this workshop focuses on introducing Python and using basic methods, so we will ignore this complication. The results we obtain will likely still be directionally meaningful but not very precise.

### Calculated Variables

Many variables in BRFSS are **derived** from raw responses — for example, BMI is computed from self-reported height and weight. These calculated variables are identifiable by a leading underscore in their names (e.g., `_BMI`). We'll encounter several of these as we work through the data.

### Data Formats
We will use the data from 2023 and 2024. For these years, there are two data formats: ASCII (fixed-width text) and SAS Transport Format (XPT). There is no CSV, Parquet, or other Python-native format offered directly. However, we can import the XPT format with `pandas.read_sas()`.

The data files can be downloaded from the BRFSS website to your hard driven. Here is the data for [2023](https://www.cdc.gov/brfss/annual_data/annual_2023.html)
and for [2024](https://www.cdc.gov/brfss/annual_data/annual_2024.html). However, they are too large to quickly upload into our Colab session. It works better to put them in your Google Drive (and better still to work with the files locally!).

## Reading and Writing Data Files

In [ ]:
# This cell is just a reminder of how you could upload a file to your session.
# Our files are too large to do it this way.
from google.colab import files
files.upload()

In [ ]:
# Uploading the files to Google Drive is much faster. After you've done so,
# run this cell to mount your drive
from google.colab import drive
drive.mount('/content/drive')

We need pandas to read in the XPT file. It is the only time we will need it in this workshop.

Although Pandas supports XPT natively,
XPT files are large and slower to read repeatedly. Even if we were to stick with using pandas instead of polars, we would immediately convert to a format that is faster to work with in Python. Parquet is best for in our case because these are ~440,000+ row datasets, and Parquet handles numeric columns (which dominate BRFSS data) very efficiently. It also preserves dtypes. Either pandas or polars can read and write parquet files.

 (NOTE: It is not necessary for you to run the cell below because you will be provided with a shortcut to the converted parquet data file. The code cell below is how the conversion was created, and it is for your reference.)

In [ ]:
import pandas as pd
import polars as pl

# You need to edit the file paths so they match the actual paths in your drive.

# Read 2023 XPT file from Drive
df23_pd = pd.read_sas("/content/drive/MyDrive/BRFSS/LLCP2023.XPT",
                    format="xport", encoding="utf-8")

# We can always go back-and-forth between pandas and polars data frames
df23_pl = pl.from_pandas(df23_pd)

# Repeat for 2024
df24_pd = pd.read_sas("/content/drive/MyDrive/BRFSS/LLCP2024.XPT",
                    format="xport", encoding="utf-8")
df24_pl = pl.from_pandas(df24_pd)

# Write parquet files back to Drive
df23_pl.write_parquet("/content/drive/MyDrive/BRFSS/brfss2023.parquet")
df24_pl.write_parquet("/content/drive/MyDrive/BRFSS/brfss2024.parquet")


Now, for every future session with this notebook file and data set, you no longer have to deal with XPT files, uploads, or pandas. Each time, you can start working by running the following cell.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# You would need to edit the path so it matches your actual path
import polars as pl
df23 = pl.read_parquet("/content/drive/MyDrive/BRFSS/brfss2023.parquet")
df24 = pl.read_parquet("/content/drive/MyDrive/BRFSS/brfss2024.parquet")

For the purposes of this workshop, to save time, you can get the parquet files from my shared [drive link](https://drive.google.com/drive/folders/1-vTbDsu5f2ccUk4knZfEZnYBwgpCWY2_?usp=sharing). Upload them to your Google Drive and run the code above, with the correct file path.

In [ ]:
# Sometimes you may want CSV format, if you want a universally readable format.

# For pandas, you would use this code:
# df.to_csv('brfss_2024.csv', index=False)

# For polars, you don't need to worry about row index, so you would use this:
# df.write_csv('brfss_2024.csv')

While CSV files are convenient for sharing, it is important to keep in mind that CSV has no data type system at all — every value is stored as plain text, and the types are entirely inferred at read time. Parquet has a rich, explicit type system that is stored as part of the file itself. When Pandas writes a Parquet file, it encodes the exact dtype of each column, and when you read it back, you get exactly those types restored — no inference, no guessing. Some of the types in BRFSS that are preserved in Parquet but lost/degraded in CSV include:

`pd.Categorical` (factor-like columns, very relevant for BRFSS coded variables)

`int8`, `int16`, `int32` vs. just `int64` (memory-important for 400k+ row surveys)

`datetime` with timezone information

For BRFSS specifically, many variables could have int8 or int16 dataypes (since values only go from 1–9 or 1–99) (e.g., 1 = Yes, 2 = No, 7 = Don't know, 9 = Refused).

However, as you will see below, we would need to specify these data types for the parquet files we create from XPT.  SAS transport files (XPT) only support two data types: character and numeric, and "numeric" in SAS is always 8-byte floating point (float64). SAS has no native integer type at all. Still, parquet has significant advantages because of how the data is stored. The CSV files would be ~5-10 times bigger.

### Plan
For the workshop and the downstream research project:  When stacking years into a single file, we need to add a `YEAR` column to each data set, and make sure column names match. We can also analyze single files, but stacking 2023, 2024 makes sense because this gives us coverage of all states plus territories and D.C.

The combining of files is complicated because the BRFSS changes calculated variable names between years — for instance, the health insurance variable is _HCVU653 in 2023 but _HCVU654 in 2024, PRIMINS1 becomes PRIMINS2, and several others have incremented suffixes. A proper append (stacking rows) would require renaming those columns to match. Let's do some initial analysis of our two data sets to determine how much work this task will entail.


## A Glance At Our Data Sets

In [ ]:
# `.shape` is the most direct equivalent of dim() in R.
# It returns a tuple of (rows, columns):

df23.shape

In [ ]:
df24.shape

In [ ]:
# You can unpack the tuple and print the components:
nrows, ncols = df23.shape
print(f"2023 BRFSS: {nrows:,} rows, {ncols} columns")

nrows, ncols = df24.shape
print(f"2024 BRFSS: {nrows:,} rows, {ncols} columns")

In [ ]:
# `.columns` is the equivalent of names() in R.
# It returns a plain Python list of strings:
df23.columns

In [ ]:
# We can show just a slice to keep the output manageable
df23.columns[:10]   # first 10 column names

In [ ]:
df24.columns[:10]   # first 10 column names

# This is standard Python list slicing. [:10] means "from the beginning
# up to (but not including) index 10." (Elements 0 through 9.)

In [ ]:
df23.dtypes

This returns a list of Polars data type objects, but without the column names it's not very informative.  More useful is `.schema`, which gives you a dictionary mapping column names to types:

In [ ]:
df23.schema

A more practical approach is to combine it with a quick check of how many columns have each type:

In [ ]:
from collections import Counter
Counter(df23.dtypes)

In [ ]:
Counter(df24.dtypes)

In [ ]:
# another way to get a quick glance at the data
df23.head()

## Compare the Two Years
We will use Python set operations on the column names to find columns unique to each year, columns in common, etc. This will help us to discover the naming discrepancies. The goal: before we combine data, we want to understand what we have.

### Common and unique column names
The `&` (intersection) and `-` (difference) operators are Python set operations. In R, the equivalents are `intersect()` and `setdiff()`.

In [ ]:
# Convert column name lists to Python sets
cols23 = set(df23.columns)
cols24 = set(df24.columns)

# Columns in both years
common = cols23 & cols24
print(f"Columns in both years: {len(common)}")

In [ ]:
# Columns only in 2023
only_23 = cols23 - cols24
print(f"Columns only in 2023: {len(only_23)}")
print(sorted(only_23))

In [ ]:
# Columns only in 2024
only_24 = cols24 - cols23
print(f"Columns only in 2024: {len(only_24)}")
print(sorted(only_24))

`sorted()` returns the results alphabetically, which helps us to eyeball the paired differences. You should see things like _HCVU653 in the 2023-only list and _HCVU654 in the 2024-only list, or PRIMINS1 vs PRIMINS2. There is an apparent pattern: CDC increments the suffix when the underlying question or derivation changes even slightly. That's a naming convention designed for provenance — it tells you the variable isn't exactly the same as last year — but it makes appending the two data sets much harder. We need the codebook to confirm which 2023 variable corresponds to which 2024 variable, and then rename to obtain matching labels. This task would be too tedious and time-consuming for the workshop, so instead we will work with separate files.

### Check which states are present
 Useful methods here: the `group_by().len()` method is the Polars equivalent of `count()` in dplyr — it groups and counts rows. The result is a new DataFrame with _STATE and len columns. `.height` is Polars for the number of rows (you could also use `.shape[0]`).

In [ ]:
# Count respondents per state
states23 = df23.group_by("_STATE").len().sort("_STATE")
states24 = df24.group_by("_STATE").len().sort("_STATE")

print(f"Distinct state codes in 2023: {states23.height}")
print(f"Distinct state codes in 2024: {states24.height}")

Now find the actual missing states:

In [ ]:
# Which FIPS codes are in one year but not the other?
fips23 = set(states23["_STATE"].to_list())
fips24 = set(states24["_STATE"].to_list())

print(f"FIPS codes in 2023 but not 2024: {fips23 - fips24}")
print(f"FIPS codes in 2024 but not 2023: {fips24 - fips23}")

### Select variables to focus on
We will pick a focused subset relevant to the research project, which is to forecast which states experience the most unmet healt needs. I confirmed from the codebooks that all six disability variables use identical names in both years: DEAF, BLIND, DECIDE, DIFFWALK, DIFFDRES, DIFFALON. The coding (1=Yes, 2=No, 7=Don't know, 9=Refused) is the same. First, define the variables we want. Building the list from named sub-lists rather than one long list is a good habit. It documents why each variable is included and makes it easy to add or remove a whole category later.

In [ ]:
# We're picking variables that tell a story about disability
# and unmet health needs, all from the BRFSS core (not optional
# modules), so they're present for every state.

# Identifiers & demographics
id_vars = ["_STATE"]

# Health status
health_vars = ["GENHLTH",     # General health: 1=Excellent ... 5=Poor
               "PHYSHLTH",    # Days physical health not good (past 30)
               "MENTHLTH"]    # Days mental health not good (past 30)

# Health care access (our "unmet needs" indicators)
access_vars = ["MEDCOST1",    # Couldn't afford to see doctor (1=Yes, 2=No)
               "CHECKUP1"]    # Time since last routine checkup

# Disability (the six standard disability questions)
# These correspond to the ACS-6 disability questions
disability_vars = ["DEAF",       # Deaf or serious difficulty hearing
                   "BLIND",      # Blind or serious difficulty seeing
                   "DECIDE",     # Difficulty concentrating/remembering/deciding
                   "DIFFWALK",   # Serious difficulty walking/climbing stairs
                   "DIFFDRES",   # Difficulty dressing or bathing
                   "DIFFALON"]   # Difficulty doing errands alone

keep_vars = id_vars + health_vars + access_vars + disability_vars
print(f"Selecting {len(keep_vars)} variables from {df23.shape[1]}")

Now we select and inspect:

In [ ]:
# Select our working subset — equivalent of dplyr::select()
sub23 = df23.select(keep_vars)
sub24 = df24.select(keep_vars)

print(f"2023 subset: {sub23.shape}")
print(f"2024 subset: {sub24.shape}")

Let's see what we have:

In [ ]:
sub23.head()

We can get a quick statistical summary.`describe()` is the Polars equivalent of R's `summary()`. It gives you count, null count, mean, std, min, max, and quartiles for each column.

In [ ]:
sub23.describe()

The `null_count` row tells you how much missing data each variable has. In Polars, genuine missing data is represented as null, which is conceptually similar to R's NA. But here's an issue with BRFSS: the survey also uses special numeric codes for missing-like responses — 7.0 for "Don't know" and 9.0 for "Refused" in the disability variables, 77 and 99 in the days-count variables, 88 meaning "None" in `PHYSHLTH/MENTHLTH`. Those aren't null, they're actual values sitting in the data. So the mean you see in .describe() is meaningless — it's averaging real responses with refusal codes. This will require some cleaning.

In [ ]:
# Look at the raw value distribution for one disability variable
sub23["DECIDE"].value_counts().sort("DECIDE")

The counts show the number of responses for 1.0 (Yes), 2.0 (No), 7.0 (Don't know), 9.0 (Refused), plus null. So we need to account for this before any analysis.

## Wrangling one year


In [ ]:
# Raw coding: 1=Yes, 2=No, 7=Don't know, 9=Refused
# We want to treat 7 and 9 as missing (null)

# For dplyr this is like
#   mutate(across(all_of(disability_vars),
#          ~na_if(na_if(.x, 7), 9)))

sub23_clean = sub23.with_columns(
    pl.when(pl.col(disability_vars).is_in([1.0, 2.0]))
      .then(pl.col(disability_vars))
      .otherwise(None)
      .name.keep()
)

Explanation: `.with_columns()` is the Polars equivalent of dplyr `mutate()`. It returns a new DataFrame with modified or added columns, leaving the original unchanged. Polars is expression-based, so rather than writing a separate statement for each column, you pass a list of column names to pl.col() and the operation applies to all of them at once. pl.col(disability_vars) selects all six disability columns simultaneously.

`pl.when().then().otherwise()` is the equivalent of `case_when()` in dplyr  (conditional logic). Here we're saying: "when the value is 1 or 2 (valid responses), keep it; otherwise, replace it with null." The .name.keep() at the end tells Polars to keep the original column names rather than generating new ones — without it, Polars would try to rename the output columns.

Now let's do the same for the other variables, which have different coding schemes:

In [ ]:
# --- Clean GENHLTH ---
# 1-5 are valid (Excellent to Poor), 7 and 9 are not
sub23_clean = sub23_clean.with_columns(
    pl.when(pl.col("GENHLTH").is_between(1, 5))
      .then(pl.col("GENHLTH"))
      .otherwise(None)
      .alias("GENHLTH")
)

# --- Clean MEDCOST1 ---
# 1=Yes, 2=No are valid; 7, 9 are not
sub23_clean = sub23_clean.with_columns(
    pl.when(pl.col("MEDCOST1").is_in([1.0, 2.0]))
      .then(pl.col("MEDCOST1"))
      .otherwise(None)
      .alias("MEDCOST1")
)

# --- Clean PHYSHLTH and MENTHLTH ---
# 1-30 = number of days, 88 = "None" (meaning 0 days)
# 77 = Don't know, 99 = Refused
sub23_clean = sub23_clean.with_columns(
    pl.when(pl.col(["PHYSHLTH", "MENTHLTH"]) == 88.0)
      .then(0.0)
      .when(pl.col(["PHYSHLTH", "MENTHLTH"]).is_between(1, 30))
      .then(pl.col(["PHYSHLTH", "MENTHLTH"]))
      .otherwise(None)
      .name.keep()
)

Notice the `PHYSHLTH/MENTHLTH` cleaning has an extra step: the code 88 doesn't mean "missing" — it means the person reported zero unhealthy days. So we recode 88 to 0 rather than to null. (It is important to look at the codebook!)

Also note the difference between `.alias("GENHLTH")` and `.name.keep()`. When you're operating on a single column, `.alias()` explicitly names the output. When you're operating on multiple columns at once (as with the disability vars or the two days variables), `.name.keep()` is more convenient because it preserves all the original names automatically.

Now we build new columns from the cleaned data.

In [ ]:
# Create "any disability" flag

# A person has a disability if they answered Yes (1) to ANY
# of the six disability questions.
#
# In dplyr: mutate(any_disability = if_any(all_of(disability_vars), ~.x == 1))

sub23_clean = sub23_clean.with_columns(
    pl.any_horizontal(
        pl.col(disability_vars) == 1.0
    ).alias("any_disability")
)

`pl.any_horizontal()` evaluates across columns within each row and returns True if any of the expressions is true. It's the equivalent of `if_any()` in dplyr. The result is a boolean column: True if the person reported at least one disability, False if they answered No to all six, and null if all six were null (they get the benefit of the doubt — if they said No to some and the rest are null, `any_horizontal` still returns False, which is correct behavior here).

In [ ]:
# Create "fair or poor health" flag
sub23_clean = sub23_clean.with_columns(
    (pl.col("GENHLTH") >= 4.0).alias("poor_health")
)

# Create "couldn't afford doctor" flag
sub23_clean = sub23_clean.with_columns(
    (pl.col("MEDCOST1") == 1.0).alias("unmet_need")
)

These are simpler — just boolean comparisons. GENHLTH is coded as: 1 = Excellent, 2 = Very good, 3 = Good, 4 = Fair, 5 = Poor (and then 7 = Don't know, 9 = Refused, which we already converted to null in the cleaning step). The parentheses around the expression are required in Polars before calling `.alias()`. MEDCOST1 is the response to: "Was there a time in the past 12 months when you needed to see a doctor but could not because you could not afford it?" Coded as 1 = Yes, 2 = No (and 7 = Don't know, 9 = Refused, which we already nulled out).

Before aggregating, it's good practice to verify the derived variables look right:

In [ ]:
# How many people report any disability?
sub23_clean["any_disability"].value_counts()

In [ ]:
# Cross-check: disability and unmet need
sub23_clean.group_by("any_disability").agg(
    pl.col("unmet_need").mean().alias("pct_unmet_need")
)

This quick cross-tab shows that people with disabilities report unmet healthcare needs at a higher rate, which is the expected finding that the project is building toward.

### State-level aggregation
We will collapse individual-level survey responses into a state-level summary. The parallel to dplyr is `group_by() |> summarize()`.

In [ ]:

# For dplyr users, this is:
#   group_by(`_STATE`) |>
#   summarize(
#     n = n(),
#     pct_disability = mean(any_disability, na.rm = TRUE),
#     pct_poor_health = mean(poor_health, na.rm = TRUE),
#     pct_unmet_need = mean(unmet_need, na.rm = TRUE),
#     avg_phys_unhealthy = mean(PHYSHLTH, na.rm = TRUE),
#     avg_mental_unhealthy = mean(MENTHLTH, na.rm = TRUE)
#   )

state_summary = sub23_clean.group_by("_STATE").agg(
    pl.len().alias("n"),
    pl.col("any_disability").mean().alias("pct_disability"),
    pl.col("poor_health").mean().alias("pct_poor_health"),
    pl.col("unmet_need").mean().alias("pct_unmet_need"),
    pl.col("PHYSHLTH").mean().alias("avg_phys_unhealthy_days"),
    pl.col("MENTHLTH").mean().alias("avg_mental_unhealthy_days"),
)

state_summary.head()

`pl.len()` counts all rows per group. It's the equivalent of `n()` in dplyr. You might also see `pl.count()` in older Polars examples, but `pl.len()` is the current preferred form.
When you take `.mean()` of a boolean column (like any_disability, poor_health, unmet_need), Polars treats True as 1 and False as 0, so the mean gives you the proportion. Nulls are excluded from the mean by default in Polars, which is equivalent to na.rm = TRUE in R. That's a nice default compared to R, where forgetting na.rm = TRUE gives you NA back.
Each `.alias()` names the output column. Without it, Polars would auto-name them things like any_disability (which actually would be fine here), but being explicit is clearer and avoids surprises when the auto-naming logic doesn't do what you expect.
Now let's make the results interpretable. The FIPS codes are meaningless to most people.

In [ ]:
# Add state names
# FIPS code to state name mapping (50 states + DC + territories)
fips_to_name = {
    1: "Alabama", 2: "Alaska", 4: "Arizona", 5: "Arkansas",
    6: "California", 8: "Colorado", 9: "Connecticut", 10: "Delaware",
    11: "District of Columbia", 12: "Florida", 13: "Georgia",
    15: "Hawaii", 16: "Idaho", 17: "Illinois", 18: "Indiana",
    19: "Iowa", 20: "Kansas", 21: "Kentucky", 22: "Louisiana",
    23: "Maine", 24: "Maryland", 25: "Massachusetts", 26: "Michigan",
    27: "Minnesota", 28: "Mississippi", 29: "Missouri", 30: "Montana",
    31: "Nebraska", 32: "Nevada", 33: "New Hampshire", 34: "New Jersey",
    35: "New Mexico", 36: "New York", 37: "North Carolina",
    38: "North Dakota", 39: "Ohio", 40: "Oklahoma", 41: "Oregon",
    42: "Pennsylvania", 44: "Rhode Island", 45: "South Carolina",
    46: "South Dakota", 47: "Tennessee", 48: "Texas", 49: "Utah",
    50: "Vermont", 51: "Virginia", 53: "Washington",
    54: "West Virginia", 55: "Wisconsin", 56: "Wyoming",
    66: "Guam", 72: "Puerto Rico", 78: "US Virgin Islands",
}

state_summary = state_summary.with_columns(
    pl.col("_STATE").cast(pl.Int16).replace_strict(
        fips_to_name, default="Unknown"
    ).alias("state_name")
)

 We first `.cast(pl.Int16)` to convert the FIPS code from float64 to integer — because our dictionary keys are integers, not floats, and Polars won't match 1.0 to 1 automatically.`.replace_strict()` does the lookup, and `default="Unknown"` handles any FIPS code we didn't include (rather than throwing an error).

 Now let's see which states have the highest unmet needs!

In [ ]:
# Top 10 states by unmet healthcare need
(state_summary
    .select("state_name", "pct_unmet_need", "pct_disability", "pct_poor_health")
    .sort("pct_unmet_need", descending=True)
    .head(10)
)

The parentheses wrapping the whole expression across multiple lines is a Python trick for line continuation without backslashes.

In [ ]:
# Top 10 states by disability prevalence
(state_summary
    .select("state_name", "pct_disability", "pct_unmet_need", "avg_phys_unhealthy_days")
    .sort("pct_disability", descending=True)
    .head(10)
)

In [ ]:
# Are disability and unmet need correlated at the state level?
# Quick visual check
print(
    state_summary.select(
        pl.corr("pct_disability", "pct_unmet_need").alias("disability_x_unmet"),
        pl.corr("pct_disability", "pct_poor_health").alias("disability_x_poor_health"),
        pl.corr("pct_unmet_need", "pct_poor_health").alias("unmet_x_poor_health"),
    )
)

`pl.corr()` computes a Pearson correlation between two columns across the whole DataFrame. You don't need a `group_by()` here because you want one correlation for all states.

There are moderately strong positive correlations. States where disability prevalence is high also tend to have more people who can't afford care and who report poor health. That's the story seed for the broader project.

In [ ]:
# Simple bar chart example
import matplotlib.pyplot as plt

top10 = (state_summary
    .sort("pct_unmet_need", descending=True)
    .head(10)
)

plt.figure(figsize=(10, 5))
plt.barh(
    top10["state_name"].to_list()[::-1],      # reverse for top-to-bottom
    top10["pct_unmet_need"].to_list()[::-1]
)
plt.xlabel("Proportion reporting unmet healthcare need")
plt.title("Top 10 States: Couldn't Afford to See a Doctor (2023 BRFSS)")
plt.tight_layout()
plt.show()